# Data Cleaning

The goal of this notebook is to transform weekly NFL player statistics into a clean player-season dataset for quarterbacks, runningbacks, receivers, and tight ends. Consequently, this notebook filters to regular-season skill-position players, aggregates weekly statistics to the season level, creates basic per-game and EPA-based features, merges useful roster information, and saves the cleaned dataset. 

## Load Data

In [20]:
import pandas as pd
import numpy as np

player_stats = pd.read_csv("../data/raw/player_stats_2016_2025.csv")
rosters = pd.read_csv("../data/raw/rosters_2016_2025.csv")
schedules = pd.read_csv("../data/raw/schedules_2016_2025.csv")

print("Player stats:", player_stats.shape)
print("Rosters:", rosters.shape)
print("Schedules:", schedules.shape)

/var/folders/7w/132kn01557v6zl_9f5zqf_t40000gn/T/ipykernel_80094/4098067335.py:4: DtypeWarning: Columns (98,114) have mixed types. Specify dtype option on import or set low_memory=False.
  player_stats = pd.read_csv("../data/raw/player_stats_2016_2025.csv")


Player stats: (182252, 115)
Rosters: (31005, 36)
Schedules: (2761, 46)


In [21]:
# player_stats columns
[col for col in player_stats.columns]

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'week',
 'season_type',
 'team',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'def_tackles_solo',
 'def_tackles_with_a

In [22]:
# Rosters columns
[col for col in rosters.columns]

['season',
 'team',
 'position',
 'depth_chart_position',
 'jersey_number',
 'status',
 'full_name',
 'first_name',
 'last_name',
 'birth_date',
 'height',
 'weight',
 'college',
 'gsis_id',
 'espn_id',
 'sportradar_id',
 'yahoo_id',
 'rotowire_id',
 'pff_id',
 'pfr_id',
 'fantasy_data_id',
 'sleeper_id',
 'years_exp',
 'headshot_url',
 'ngs_position',
 'week',
 'game_type',
 'status_description_abbr',
 'football_name',
 'esb_id',
 'gsis_it_id',
 'smart_id',
 'entry_year',
 'rookie_year',
 'draft_club',
 'draft_number']

## Aggregate Skill Position Weekly Stats to Season Level

In [23]:
#filter to regular season skill players

skill_positions = ["QB", "RB", "WR", "TE"]

skill_weekly = player_stats[
    (player_stats["position"].isin(skill_positions)) &
    (player_stats["season_type"] == "REG")
].copy()

skill_weekly.shape


(56529, 115)

In [24]:
#main stat columns available for skill players
[col for col in skill_weekly.columns if any(word in col for word in [
    "passing", "rushing", "receiving", "targets", "receptions",
    "carries", "attempts", "interceptions", "fantasy"
])]

['attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'def_interceptions',
 'fantasy_points',
 'fantasy_points_ppr']

In [25]:
# Create season level dataset by summing weekly stats

sum_cols = [
    "completions",
    "attempts",
    "passing_yards",
    "passing_tds",
    "interceptions",
    "sacks",
    "sack_yards",
    "passing_air_yards",
    "passing_yards_after_catch",
    "passing_first_downs",
    "passing_epa",
    "passing_2pt_conversions",
    
    "carries",
    "rushing_yards",
    "rushing_tds",
    "rushing_fumbles_lost"
    "rushing_first_downs",
    "rushing_epa",
    "rushing_2pt_conversions",
    
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_fumbles_lost",
    "receiving_tds",
    "receiving_air_yards",
    
    "receiving_yards_after_catch",
    "receiving_first_downs",
    "receiving_epa",
    "receiving_2pt_conversions",
    
    "target_share",
    "air_yards_share",
    "wopr",
    "fantasy_points",
    "fantasy_points_ppr"
]

# Keep only columns that actually exist
sum_cols = [col for col in sum_cols if col in skill_weekly.columns]

skill_season = (
    skill_weekly
    .groupby(
        ["season", "player_id", "player_name", "player_display_name", "position", "team"],
        as_index=False
    )[sum_cols]
    .sum()
)

skill_season.head()

,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,target_share,air_yards_share,wopr,fantasy_points,fantasy_points_ppr
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,0,0,0,0.000000,0,0.000000,0.000000,0.000000,258.56,258.56
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,808,261,36,34.491710,2,2.453040,2.785009,5.629066,110.50,177.50
2,2016,00-0020337,S.Smith Sr.,Steve Smith,WR,BAL,0,0,0,0,...,65,20,3,3.455455,0,0.102041,0.217391,0.305235,3.40,6.40
3,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,0,0,0,0.000000,0,0.000000,0.000000,0.000000,332.32,332.32
4,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,0,0,0,0.000000,0,0.000000,0.000000,0.000000,10.18,10.18


In [26]:
# Adding games played
games_played = (
    skill_weekly
    .groupby(["season", "player_id"], as_index=False)["week"]
    .nunique()
    .rename(columns={"week": "games_played"})
)

skill_season = skill_season.merge(
    games_played,
    on=["season", "player_id"],
    how="left"
)

skill_season.head()

,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,target_share,air_yards_share,wopr,fantasy_points,fantasy_points_ppr,games_played
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,0,0,0.000000,0,0.000000,0.000000,0.000000,258.56,258.56,12
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,261,36,34.491710,2,2.453040,2.785009,5.629066,110.50,177.50,14
2,2016,00-0020337,S.Smith Sr.,Steve Smith,WR,BAL,0,0,0,0,...,20,3,3.455455,0,0.102041,0.217391,0.305235,3.40,6.40,14
3,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,0,0,0.000000,0,0.000000,0.000000,0.000000,332.32,332.32,16
4,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,0,0,0.000000,0,0.000000,0.000000,0.000000,10.18,10.18,3


In [27]:
# Additional season level features

# -----------------------------
# General total production
# -----------------------------
skill_season["total_yards"] = (
    skill_season.get("passing_yards", 0) +
    skill_season.get("rushing_yards", 0) +
    skill_season.get("receiving_yards", 0)
)

skill_season["total_epa"] = (
    skill_season.get("passing_epa", 0) +
    skill_season.get("rushing_epa", 0) +
    skill_season.get("receiving_epa", 0)
)

skill_season["total_tds"] = (
    skill_season.get("passing_tds", 0) +
    skill_season.get("rushing_tds", 0) +
    skill_season.get("receiving_tds", 0)
)

skill_season["yards_per_game"] = (
    skill_season["total_yards"] / skill_season["games_played"]
)

skill_season["epa_per_game"] = (
    skill_season["total_epa"] / skill_season["games_played"]
)

skill_season["tds_per_game"] = (
    skill_season["total_tds"] / skill_season["games_played"]
)

# -----------------------------
# RB / WR / TE scrimmage features
# -----------------------------
skill_season["scrimmage_touches"] = (
    skill_season.get("carries", 0) +
    skill_season.get("receptions", 0)
)

skill_season["scrimmage_yards"] = (
    skill_season.get("rushing_yards", 0) +
    skill_season.get("receiving_yards", 0)
)

skill_season["scrimmage_tds"] = (
    skill_season.get("rushing_tds", 0) +
    skill_season.get("receiving_tds", 0)
)

skill_season["scrimmage_yards_per_game"] = (
    skill_season["scrimmage_yards"] / skill_season["games_played"]
)

skill_season["scrimmage_epa"] = (
    skill_season.get("rushing_epa", 0) +
    skill_season.get("receiving_epa", 0)
)

skill_season["scrimmage_epa_per_game"] = (
    skill_season["scrimmage_epa"] / skill_season["games_played"]
)

skill_season["scrimmage_touches_per_game"] = (
    skill_season["scrimmage_touches"] / skill_season["games_played"]
)

skill_season["yards_per_scrimmage_touch"] = (
    skill_season["scrimmage_yards"] /
    skill_season["scrimmage_touches"].replace(0, np.nan)
)


# -----------------------------
# QB-specific features
# -----------------------------
skill_season["qb_plays"] = (
    skill_season.get("attempts", 0) +
    skill_season.get("carries", 0)
)

skill_season["qb_total_yards"] = (
    skill_season.get("passing_yards", 0) +
    skill_season.get("rushing_yards", 0)
)

skill_season["qb_epa"] = (
    skill_season.get("passing_epa", 0) +
    skill_season.get("rushing_epa", 0)
)

skill_season["qb_total_tds"] = (
    skill_season.get("passing_tds", 0) +
    skill_season.get("rushing_tds", 0)
)

skill_season["qb_yards_per_play"] = (
    skill_season["qb_total_yards"] /
    skill_season["qb_plays"].replace(0, np.nan)
)

skill_season["qb_yards_per_game"] = (
    skill_season["qb_total_yards"] / skill_season["games_played"]
)

skill_season["qb_epa_per_game"] = (
    skill_season["qb_epa"] / skill_season["games_played"]
)

skill_season["qb_tds_per_game"] = (
    skill_season["qb_total_tds"] / skill_season["games_played"]
)


skill_season.head()

,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,scrimmage_touches_per_game,yards_per_scrimmage_touch,qb_plays,qb_total_yards,qb_epa,qb_total_tds,qb_yards_per_play,qb_yards_per_game,qb_epa_per_game,qb_tds_per_game
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,2.333333,2.285714,460,3618,140.632333,28,7.865217,301.500000,11.719361,2.333333
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,4.785714,11.417910,0,0,0.000000,0,NaN,0.000000,0.000000,0.000000
2,2016,00-0020337,S.Smith Sr.,Steve Smith,WR,BAL,0,0,0,0,...,0.214286,11.333333,0,0,0.000000,0,NaN,0.000000,0.000000,0.000000
3,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,1.437500,0.869565,696,5228,105.672159,39,7.511494,326.750000,6.604510,2.437500
4,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,1.666667,1.000000,40,247,4.260199,0,6.175000,82.333333,1.420066,0.000000


In [28]:
# Retain only relevant columns from rosters
roster_cols = [
    "season",
    "gsis_id",
    "birth_date",
    "height",
    "weight",
    "years_exp",
    "entry_year",
    "rookie_year",
    "draft_club",
    "draft_number",
    "college"
]

rosters_trimmed = rosters[roster_cols].drop_duplicates()

# Merge season level stats with roster info

skill_season = skill_season.merge(
    rosters_trimmed,
    left_on=["season", "player_id"],
    right_on=["season", "gsis_id"],
    how="left"
)

skill_season.head()

,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,gsis_id,birth_date,height,weight,years_exp,entry_year,rookie_year,draft_club,draft_number,college
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,00-0019596,1977-08-03,76.0,225.0,16.0,2000.0,2000.0,NE,NaN,Michigan
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,00-0020337,1979-05-12,69.0,195.0,NaN,NaN,NaN,NaN,NaN,Utah
2,2016,00-0020337,S.Smith Sr.,Steve Smith,WR,BAL,0,0,0,0,...,00-0020337,1979-05-12,69.0,195.0,NaN,NaN,NaN,NaN,NaN,Utah
3,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,00-0020531,1979-01-15,72.0,209.0,15.0,2001.0,2001.0,SD,NaN,Purdue
4,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,00-0020679,1980-01-09,75.0,230.0,14.0,2002.0,2002.0,NaN,NaN,Maryland


In [29]:
# Adding age
skill_season["birth_date"] = pd.to_datetime(skill_season["birth_date"], errors="coerce")

skill_season["age"] = (
    skill_season["season"] -
    skill_season["birth_date"].dt.year
)

skill_season[["player_display_name", "season", "position", "age", "years_exp"]].head()

,player_display_name,season,position,age,years_exp
0,Tom Brady,2016,QB,39,16.0
1,Steve Smith,2016,WR,37,NaN
2,Steve Smith,2016,WR,37,NaN
3,Drew Brees,2016,QB,37,15.0
4,Shaun Hill,2016,QB,36,14.0


In [30]:
skill_season.head()


,season,player_id,player_name,player_display_name,position,team,completions,attempts,passing_yards,passing_tds,...,birth_date,height,weight,years_exp,entry_year,rookie_year,draft_club,draft_number,college,age
0,2016,00-0019596,T.Brady,Tom Brady,QB,NE,291,432,3554,28,...,1977-08-03,76.0,225.0,16.0,2000.0,2000.0,NE,NaN,Michigan,39
1,2016,00-0020337,S.Smith,Steve Smith,WR,BAL,0,0,0,0,...,1979-05-12,69.0,195.0,NaN,NaN,NaN,NaN,NaN,Utah,37
2,2016,00-0020337,S.Smith Sr.,Steve Smith,WR,BAL,0,0,0,0,...,1979-05-12,69.0,195.0,NaN,NaN,NaN,NaN,NaN,Utah,37
3,2016,00-0020531,D.Brees,Drew Brees,QB,NO,471,673,5208,37,...,1979-01-15,72.0,209.0,15.0,2001.0,2001.0,SD,NaN,Purdue,37
4,2016,00-0020679,S.Hill,Shaun Hill,QB,MIN,19,35,242,0,...,1980-01-09,75.0,230.0,14.0,2002.0,2002.0,NaN,NaN,Maryland,36


In [31]:
# check for duplicates
# duplicates could indicate that a player played for multiple teams in a season
# this is okay for now but will need to be handled eventually
duplicates = (
    skill_season
    .groupby(["season", "player_id"])
    .size()
    .reset_index(name="num_rows")
    .sort_values("num_rows", ascending=False)
)

duplicates.head(20)

,season,player_id,num_rows
4835,2024,00-0035216,4
2988,2021,00-0032208,3
1763,2019,00-0029632,3
1243,2018,00-0029848,3
3898,2022,00-0036383,3
1911,2019,00-0032423,3
3434,2021,00-0036698,3
64,2016,00-0026373,3
3962,2022,00-0036877,2
213,2016,00-0029683,2


In [32]:
# missing values in roster variables
# draft_number has a high missing value percentage, could be due to many undrafted players
skill_season[["birth_date", "age", "years_exp", "height", "weight", "draft_number"]].isna().mean()

birth_date      0.000000
age             0.000000
years_exp       0.004238
height          0.000000
weight          0.000000
draft_number    0.368378
dtype: float64

In [33]:
# Checking that features makes sense
# Should see mostly QBs at the top for total yards
skill_season[
    ["player_display_name", "season", "position", "team", "games_played",
     "total_yards", "total_tds", "scrimmage_yards", "scrimmage_touches",
     "qb_total_yards", "qb_plays"]
].sort_values("total_yards", ascending=False).head(20)

,player_display_name,season,position,team,games_played,total_yards,total_tds,scrimmage_yards,scrimmage_touches,qb_total_yards,qb_plays
3839,Patrick Mahomes,2022,QB,KC,17,5614,45,364,62,5608,709
2992,Tom Brady,2021,QB,TB,17,5397,45,81,28,5397,747
1622,Patrick Mahomes,2018,QB,KC,16,5369,52,272,60,5369,640
1927,Jameis Winston,2019,QB,TB,16,5359,34,250,59,5359,685
3538,Justin Herbert,2021,QB,LAC,17,5316,41,302,63,5316,735
2628,Deshaun Watson,2020,QB,HOU,16,5267,36,444,90,5267,634
3,Drew Brees,2016,QB,NO,16,5228,39,20,23,5228,696
1159,Ben Roethlisberger,2018,QB,PIT,16,5226,37,97,32,5227,706
3250,Patrick Mahomes,2021,QB,KC,17,5220,39,381,66,5220,724
2047,Dak Prescott,2019,QB,DAL,16,5179,33,277,52,5179,648


In [34]:
# Should be Non-QBs at the top, specifically mostly RBs for scrimmage yards
skill_season[
    skill_season["position"].isin(["RB", "WR", "TE"])
][
    ["player_display_name", "season", "position", "team", "games_played",
     "scrimmage_yards", "scrimmage_touches", "yards_per_scrimmage_touch"]
].sort_values("scrimmage_yards", ascending=False).head(20)


,player_display_name,season,position,team,games_played,scrimmage_yards,scrimmage_touches,yards_per_scrimmage_touch
2067,Christian McCaffrey,2019,RB,CAR,16,2392,403,5.935484
5873,Bijan Robinson,2025,RB,ATL,17,2298,366,6.278689
5036,Saquon Barkley,2024,RB,PHI,16,2283,378,6.039683
3504,Jonathan Taylor,2021,RB,IND,17,2171,372,5.836022
2557,Derrick Henry,2020,RB,TEN,16,2141,397,5.392947
5549,Christian McCaffrey,2025,RB,SF,17,2126,413,5.147700
460,David Johnson,2016,RB,ARI,16,2118,373,5.678284
4939,Derrick Henry,2024,RB,BAL,17,2114,344,6.145349
956,Todd Gurley,2017,RB,LA,15,2093,343,6.102041
4010,Josh Jacobs,2022,RB,LV,17,2053,393,5.223919


In [35]:
skill_season.to_csv("../data/processed/skill_player_seasons_2016_2025.csv", index=False)